# Week 38, Day 5: The ventral stream: selectivity and invariance

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w038/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

### Exercise: Pooling buys tolerance

Write C1(im, pool=4): for each of the four S1 orientation maps, take the maximum over pool by pool neighbourhoods (maximum_filter) and keep every second pixel. Then compare the cosine similarity of S1 and of C1 features for a shape and the same shape shifted by 2 pixels.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve, maximum_filter
r = np.random.default_rng(1999); S = 28

def shape(kind, cx, cy, noise=0.1):
    im = np.zeros((S, S)); L = 4
    if kind == "+": im[cy, cx - L:cx + L + 1] = 1; im[cy - L:cy + L + 1, cx] = 1
    if kind == "T": im[cy - L, cx - L:cx + L + 1] = 1; im[cy - L:cy + L + 1, cx] = 1
    if kind == "B": im[cy + L, cx - L:cx + L + 1] = 1; im[cy - L:cy + L + 1, cx] = 1     # upside-down T
    if kind == "X":
        for k in range(2 * L + 1): im[cy - L + k, cx - L + k] = 1; im[cy - L + k, cx + L - k] = 1
    return im + noise * r.normal(size=(S, S))

yy, xx = np.mgrid[-3:4, -3:4]
def gab(th):
    t = np.deg2rad(th); xr = xx * np.cos(t) + yy * np.sin(t); yr = -xx * np.sin(t) + yy * np.cos(t)
    g = np.exp(-xr ** 2 / 2 - yr ** 2 / 8) * np.cos(2 * np.pi * xr / 5); return g - g.mean()
bank = [gab(t) for t in (0, 45, 90, 135)]

def S1(im):
    return np.stack([np.abs(convolve(im, g)) for g in bank])

def C1(im, pool=4):
    ...

cos = lambda a, b: np.dot(a.ravel(), b.ravel()) / (np.linalg.norm(a) * np.linalg.norm(b))
a, b = shape("+", 12, 14, noise=0), shape("+", 14, 14, noise=0)
sim_s1, sim_c1 = cos(S1(a), S1(b)), cos(C1(a), C1(b))
print(f"similarity after a 2-pixel shift: S1 {sim_s1:.2f}, C1 {sim_c1:.2f}")
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
ax[0].imshow(a, cmap="gray"); ax[1].imshow(S1(a)[2], cmap="magma"); ax[2].imshow(C1(a)[2], cmap="magma")
for x_, t_ in zip(ax, ["image", "S1, one orientation", "C1, same orientation"]): x_.set_title(t_); x_.axis("off")
plt.show()

In [ ]:
import numpy as np
def test_tolerance():
    "after a 2-pixel shift, C1 features stay more similar than S1 features"
    assert sim_c1 > sim_s1 + 0.1 and sim_c1 > 0.8

test_tolerance()
print("All checks passed")

### Exercise: Generalising to new positions

Write global_pool(im): the maximum of each of the four S1 maps over the whole image (a vector of 4). Then compare the two feature sets on the left-to-right test.

In [ ]:
def dataset(classes, n, xs):
    ims, ys = [], []
    for k, c in enumerate(classes):
        for _ in range(n):
            ims.append(shape(c, r.integers(*xs), r.integers(6, S - 6))); ys.append(k)
    return ims, np.array(ys)

def nearest_centroid(Ftr, ytr, Fte, yte):
    mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-9; A, B = (Ftr - mu) / sd, (Fte - mu) / sd
    cent = np.array([A[ytr == k].mean(0) for k in np.unique(ytr)])
    return np.mean(np.argmin(((B[:, None] - cent[None]) ** 2).sum(-1), 1) == yte)

def global_pool(im):
    ...

tr, ytr = dataset(["+", "X"], 40, (6, 11)); te, yte = dataset(["+", "X"], 40, (17, 22))
acc = {"pixels": nearest_centroid(np.array([i.ravel() for i in tr]), ytr, np.array([i.ravel() for i in te]), yte),
       "global pooling": nearest_centroid(np.array([global_pool(i) for i in tr]), ytr, np.array([global_pool(i) for i in te]), yte)}
print({k: f"{v:.0%}" for k, v in acc.items()}, "correct on shapes at new positions")

In [ ]:
import numpy as np
def test_pixels():
    "pixels generalise to new positions at chance (below 65%)"
    assert acc["pixels"] < 0.65

def test_global():
    "globally pooled features generalise above 95%"
    assert acc["global pooling"] > 0.95

test_pixels()
test_global()
print("All checks passed")

### Exercise: T against upside-down T

Write C2(im): for each template, compute the squared distance between it and every 4 by 4 patch of the image's C1 maps (all four orientations), and return exp(-minimum distance / template energy) for each template, a vector with one value per template.

In [ ]:
def patches(m, size=4):
    H = m.shape[1]
    return np.array([m[:, i:i + size, j:j + size] for i in range(H - size + 1) for j in range(H - size + 1)])

templates = []
for kind in ("T", "B"):                          # templates from one centred example of each shape
    ps = patches(C1(shape(kind, 14, 14, noise=0)))
    for idx in np.argsort(ps.sum(axis=(1, 2, 3)))[-6:]:
        templates.append(ps[idx])

def C2(im):
    ps = patches(C1(im))
    ...

tr, ytr = dataset(["T", "B"], 40, (6, 11)); te, yte = dataset(["T", "B"], 40, (17, 22))
acc_tb = {"global pooling": nearest_centroid(np.array([global_pool(i) for i in tr]), ytr, np.array([global_pool(i) for i in te]), yte),
          "C2": nearest_centroid(np.array([C2(i) for i in tr]), ytr, np.array([C2(i) for i in te]), yte)}
print({k: f"{v:.0%}" for k, v in acc_tb.items()}, "correct for T against upside-down T at new positions")

In [ ]:
import numpy as np
def test_global_fails():
    "global orientation pooling is near chance (below 65%)"
    assert acc_tb["global pooling"] < 0.65

def test_c2():
    "C2 conjunction features exceed 90% on new positions"
    assert acc_tb["C2"] > 0.9

test_global_fails()
test_c2()
print("All checks passed")